The equation $u \left( t, x \right) = e^{- \alpha \pi^2 t} \sin \left( \pi x \right) $ satisfies

1. PDE $$
u_t(t, x) = - \sin(πx)α π^2 e^{- α π^2 t}
$$
$$
u_{xx}(t, x) = - \pi^2 \sin(\pi x) e^{- α\pi^2 t}
$$
This gives us
$$
u_t(t,x) = u_{xx}(t,x) α
$$

or
$$
u_t(t,x) - u_{xx}(t,x) α = 0
$$
2. $$$$
3. $$$$

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
import torch.autograd as autograd
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

In [ ]:
class HeatPINN(nn.Module):
  def __init__(self):
    super().__init__()

    self.seq = nn.Sequential(nn.Linear(2, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 1))

  def forward(self, x):
    return self.seq(x)

model = HeatPINN()

1. Dimension is 2
2. Shape is `(N, 1)`
3. Because each input has shape of `(1, )` so therefore it goes from `(N, 2)` to `(N, 1)`

In [ ]:
class Dataset_PDE(Dataset):
  def __init__(self, len):
    self.data = torch.rand(len, 2, requires_grad=True)

  def __len__(self):
    return self.data.shape[0]

  def __getitem__(self, idx):
    return self.data[idx]

dataset_train_PDE = Dataset_PDE(500)

In [ ]:
dataloader_PDE = DataLoader(dataset_train_PDE, batch_size = 32, shuffle= True)

In [ ]:
num_samples = 101

dataset_train_IC = torch.stack((torch.zeros(num_samples), torch.linspace(0, 1, num_samples)), dim = 1)

dataset_train_IC.requires_grad = True

print(dataset_train_IC.requires_grad)
print(dataset_train_IC.shape)

u_IC = torch.sin(dataset_train_IC[:,1].view(-1, 1) * torch.pi)

u_IC.requires_grad_(True)

print(u_IC.requires_grad)
print(u_IC.shape)

True
torch.Size([101, 2])
True
torch.Size([101, 1])


In [ ]:
num_samples = 202

data_0 = torch.stack([torch.linspace(0, 1, num_samples//2), torch.zeros(num_samples//2)], dim = 1)
data_1 = torch.stack([torch.linspace(0, 1, num_samples//2), torch.ones(num_samples//2)], dim = 1)

dataset_train_BC = torch.cat([data_0, data_1])

dataset_train_BC.requires_grad_(True)

print(dataset_train_BC.requires_grad)
print(dataset_train_BC.shape)

True
torch.Size([202, 2])


In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
p = torch.tensor(1.0, requires_grad=True)
q = torch.tensor(2.0, requires_grad = True)

f = p**2 + q**3 + p * q**2

f_p = autograd.grad(f, p, create_graph=True)[0]
print(f_p)

f_q = autograd.grad(f, q, create_graph=True)[0]
print(f_q)

f_pp = autograd.grad(f_p, p, create_graph=True)[0]
print(f_pp)

f_qq = autograd.grad(f_q, q, create_graph=True)[0]
print(f_qq)

f_pq = autograd.grad(f_p, q, create_graph=True)[0]
print(f_pq)

f_qqq = autograd.grad(f_qq, q, create_graph=True)[0]
print(f_qqq)

x = torch.linspace(0, 1, 10, requires_grad=True)

g = x**2
g_x = autograd.grad(g.sum(), x, create_graph=True)[0]

print(g_x)
print(g_x.shape)

g_xx = autograd.grad(g_x.sum(), x, create_graph = True)[0]
print(g_xx)
print(g_xx.shape)

tensor(6., grad_fn=<AddBackward0>)
tensor(16., grad_fn=<AddBackward0>)
tensor(2., grad_fn=<MulBackward0>)
tensor(14., grad_fn=<AddBackward0>)
tensor(4., grad_fn=<MulBackward0>)
tensor(6., grad_fn=<AddBackward0>)
tensor([0.0000, 0.2222, 0.4444, 0.6667, 0.8889, 1.1111, 1.3333, 1.5556, 1.7778,
        2.0000], grad_fn=<MulBackward0>)
torch.Size([10])
tensor([2., 2., 2., 2., 2., 2., 2., 2., 2., 2.], grad_fn=<MulBackward0>)
torch.Size([10])


In [ ]:
alpha=0.1
num_epochs = 1000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for epoch in tqdm(range(num_epochs)):
  model.train()

  model.to(device)
  for batch_PDE in dataloader_PDE:
    batch_PDE = batch_PDE.to(device)
    optimizer.zero_grad()
    U_PDE = model(batch_PDE)
    U_t_and_x_PDE = autograd.grad(torch.sum(U_PDE), batch_PDE, create_graph=True)[0]